In [16]:
import os
import numpy as np
import pandas as pd
import networkx as nx
from CTDNE.ctdne import CTDNE


def run_ctdne_to_npy(
    file_path,
    dimensions=64,
    walk_length=30,
    num_walks=200,
    workers=4,
    window=10,
    min_count=1,
    batch_words=4,
):
    df = pd.read_csv(file_path)
    df = df.sort_values("timestamp").reset_index(drop=True)

    G = nx.MultiGraph()
    for row in df.itertuples(index=False):
        u = int(row.source)
        v = int(row.destination)
        t = row.timestamp
        G.add_edge(u, v, time=float(t))

    ctdne_model = CTDNE(
        G,
        dimensions=dimensions,
        walk_length=walk_length,
        num_walks=num_walks,
        workers=workers
    )

    model = ctdne_model.fit(
        window=window,
        min_count=min_count,
        batch_words=batch_words
    )

    nodes_str = model.wv.index_to_key
    emb = model.wv.vectors.astype(np.float32)

    nodes = []
    for x in nodes_str:
        try:
            nodes.append(int(x))
        except:
            raise ValueError(f"Node id {x} cannot be converted to int.")

    max_node_id = int(max(df["source"].max(), df["destination"].max()))
    num_nodes = max_node_id + 1
    dim = emb.shape[1]

    node_features = np.zeros((num_nodes, dim), dtype=np.float32)

    for i, node in enumerate(nodes):
        if 0 <= node < num_nodes:
            node_features[node] = emb[i]

    base_name = os.path.splitext(os.path.basename(file_path))[0]
    os.makedirs("pretrain", exist_ok=True)
    out_path = os.path.join(base_name + ".npy")

    np.save(out_path, node_features)

    print(f"Saved embeddings to: {out_path}")
    print(f"node_features.shape = {node_features.shape}")

    return node_features


_ = run_ctdne_to_npy("../syn_data/p0.8_mu0.2_4.csv")

Generating walks (CPU: 4): 100%|██████████| 50/50 [00:00<00:00, 230.19it/s]


Saved embeddings to: p0.8_mu0.2_4.npy
node_features.shape = (50, 64)


In [11]:
import os
import glob
import numpy as np
import pandas as pd
import networkx as nx
from CTDNE.ctdne import CTDNE

INPUT_DIR = "../syn_data"
OUTPUT_DIR = "../pretrain"

dimensions = 64
walk_length = 30
num_walks = 200
workers = 4
window = 10
min_count = 1
batch_words = 4

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.csv")))
print(f"Found {len(csv_files)} csv files.")

for file_path in csv_files:
    print(f"\nProcessing: {file_path}")

    df = pd.read_csv(file_path)
    df = df.sort_values("timestamp").reset_index(drop=True)

    G = nx.MultiGraph()
    for row in df.itertuples(index=False):
        u = int(row.source)
        v = int(row.destination)
        t = float(row.timestamp)
        G.add_edge(u, v, time=t)

    ctdne_model = CTDNE(
        G,
        dimensions=dimensions,
        walk_length=walk_length,
        num_walks=num_walks,
        workers=workers
    )

    model = ctdne_model.fit(
        window=window,
        min_count=min_count,
        batch_words=batch_words
    )

    nodes_str = model.wv.index_to_key
    emb = model.wv.vectors.astype(np.float32)

    nodes = np.array([int(x) for x in nodes_str], dtype=np.int64)
    order = np.argsort(nodes)
    emb = emb[order]

    base = os.path.splitext(os.path.basename(file_path))[0]
    out_path = os.path.join(OUTPUT_DIR, base + ".npy")

    np.save(out_path, emb)
    print(f"Saved to: {out_path}, shape = {emb.shape}")

print("\nAll files done.")

Found 100 csv files.

Processing: ../syn_data/p0.85_mu0.05_1.csv


Generating walks (CPU: 1): 100%|██████████| 50/50 [00:00<00:00, 231.81it/s]


Saved to: ../pretrain/p0.85_mu0.05_1.npy, shape = (50, 64)

Processing: ../syn_data/p0.85_mu0.05_2.csv


Generating walks (CPU: 4): 100%|██████████| 50/50 [00:00<00:00, 226.79it/s]


Saved to: ../pretrain/p0.85_mu0.05_2.npy, shape = (50, 64)

Processing: ../syn_data/p0.85_mu0.05_3.csv


...

Saved to: ../pretrain/p1.0_mu0.2_2.npy, shape = (50, 64)

Processing: ../syn_data/p1.0_mu0.2_3.csv


Generating walks (CPU: 4): 100%|██████████| 50/50 [00:00<00:00, 161.68it/s]


Saved to: ../pretrain/p1.0_mu0.2_3.npy, shape = (50, 64)

Processing: ../syn_data/p1.0_mu0.2_4.csv


Generating walks (CPU: 2): 100%|██████████| 50/50 [00:00<00:00, 164.39it/s]


Saved to: ../pretrain/p1.0_mu0.2_4.npy, shape = (50, 64)

All files done.
